In [ ]:
    ############    #############   Liveness vs Readiness   #############   ##############   

 =>  Liveness ('/health' or '/healthz'): is the process itself alive and able to respond at
       all? Should almost never fail unless the process is truly stuck -- a deployment
       platform (Kubernetes) restarts the container when this fails.

 =>  Readiness ('/ready' or '/readyz'): is the process ready to serve REAL traffic right
       now -- are its dependencies (database, cache, downstream APIs) actually reachable?
       A platform stops ROUTING traffic to an instance that fails readiness, without
       restarting it.

 =>  Conflating the two is a classic outage cause: if a slow database makes '/health' fail,
       Kubernetes will restart every instance at once -- the exact opposite of what you want
       when the real problem is 'give the database a moment to recover'.


In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI()

class FakeDb:
    def __init__(self):
        self.is_reachable = True

db = FakeDb()

@app.get("/healthz")
def liveness():
    # only checks: is this process able to respond at all?
    return {"status": "alive"}

@app.get("/readyz")
def readiness():
    checks = {"database": db.is_reachable}
    all_ok = all(checks.values())
    status_code = 200 if all_ok else 503
    from fastapi.responses import JSONResponse
    return JSONResponse(status_code=status_code, content={"status": "ready" if all_ok else "not ready", "checks": checks})

client = TestClient(app)
print("liveness:", client.get("/healthz").json())
print("readiness (db up):", client.get("/readyz").status_code, client.get("/readyz").json())

db.is_reachable = False
print("readiness (db down):", client.get("/readyz").status_code, client.get("/readyz").json())


In [ ]:
 =>  Liveness stays 'alive' the whole time -- the process itself never stopped
       responding, even while the database was down.

 =>  Readiness correctly flips to 503 when the database becomes unreachable -- this tells
       the load balancer to stop sending traffic here, without triggering a pointless
       container restart.


In [ ]:
    ############    #############   Hands-on Lab Checklist   #############   ##############   

 =>  [ ] Add a real dependency check to /readyz (e.g. a lightweight 'SELECT 1' against a
           real Postgres connection, with a short timeout).

 =>  [ ] Configure a Kubernetes Deployment's livenessProbe and readinessProbe against these
           two different endpoints (Phase 8 covers this in depth).


In [ ]:
    ############    #############   Common Pitfalls   #############   ##############   

 =>  Making /healthz check downstream dependencies -- a flaky third-party API now causes
       your OWN healthy process to get restarted repeatedly, for no good reason.

 =>  Readiness checks that are too slow/expensive -- they're polled frequently (every few
       seconds); an expensive check adds constant background load.
